## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"


### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Install required libraries
!pip install -q --disable-pip-version-check \
    langchain==0.3.27 \
    langchain-community==0.3.27 \
    langchain-openai==0.3.30 \
    chromadb==1.0.15 \
    pymupdf==1.26.3 \
    tiktoken==0.9.0 \
    datasets==4.0.0 \
    evaluate==0.4.5

In [ ]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data
import requests  # type: ignore                                                 # Make HTTP requests (e.g., API calls); ignore type checker

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
# from langchain_community.document_loaders import PyPDFLoader                    # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore

from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
# Load the JSON file and extract values
file_name = "Config.json"                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()

In [ ]:
# Define response function
def response(user_prompt, max_tokens= 512, temperature= 0.7, top_p=1):
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
    )
    return completion.choices[0].message.content

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"

In [ ]:
question_1_answer = response(question_1)

In [ ]:
print(question_1_answer)

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
question_2_answer = response(question_2)
print(question_2_answer)

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
question_3_answer = response(question_3)
print(question_3_answer)

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
question_4_answer = response(question_4)
print(question_4_answer)

In [ ]:
#create a pandas data frame
results_baseline = pd.DataFrame({
    'Question': [question_1, question_2, question_3, question_4],
    'Answer': [question_1_answer, question_2_answer, question_3_answer, question_4_answer]
})
#display the dataframe
results_baseline.head()

## Question Answering using LLM with Prompt Engineering

In [ ]:
# Prompt to enhance the models answer
system_prompt = """
You are a clinical decision support assistant. Your task is to assist healthcare professionals by retrieving and synthesizing relevant, up-to-date, and trusted medical knowledge from authorized medical manuals, peer-reviewed research papers, and clinical guidelines. When provided with a clinical query, patient case description, or medical scenario:

Retrieve only relevant information from trusted medical sources within the knowledge base.
Synthesize the retrieved information into a clear, concise, and clinically useful response.
Prioritize evidence-based guidance and current standards of care.
Highlight critical considerations for time-sensitive or emergency situations.
Avoid speculation or unsupported claims.
If sufficient information is not available in the retrieved documents, explicitly state:
  "Insufficient information in the knowledge base to provide a reliable recommendation."

Return the output in the following structured JSON format only:
{
  "ClinicalSummary": "Concise summary of the clinical issue",
  "KeyFindings": ["bullet_point_1", "bullet_point_2", "bullet_point_3"],
  "RecommendedActions": ["action_1", "action_2"],
  "EvidenceSources": ["source_1", "source_2"],
  "UrgencyLevel": "Low | Moderate | High | Emergency"
}
"""

In [ ]:
#Define function to get response with Engineered Prompt
def response(system_prompt, user_prompt, max_tokens= 512, temperature= 0.7, top_p=1):
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
    )
    return completion.choices[0].message.content

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt_response1 = response(system_prompt, question_1)
prompt_response1

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt_response2 = response(system_prompt, question_2)
prompt_response2

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt_response3 = response(system_prompt, question_3)
prompt_response3

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt_response4 = response(system_prompt, question_4)
prompt_response4

In [ ]:
#Update the results
results_baseline['promt_response'] = [prompt_response1, prompt_response2, prompt_response3, prompt_response4]

In [ ]:
#dispaly the results
results_baseline.head()

## Data Preparation for RAG

### Loading the Data

In [ ]:
medical_diagnosis_manual_path = "medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(medical_diagnosis_manual_path)
manual = pdf_loader.load()

### Data Overview

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

### Data Chunking

In [ ]:
#Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1500, #chunk size
    chunk_overlap= 300 #chunk overlap
)

In [ ]:
#Split pdf into chunks
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
#get legnth of the chunks
len(document_chunks)

### Embedding

In [ ]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,
    openai_api_base=OPENAI_API_BASE,
    chunk_size=200 # Explicitly setting chunk size for embedding requests to avoid exceeding API token limits
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))

# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

### Vector Database

In [ ]:
#vector database directory
out_dir = 'chroma_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

### Retriever

In [ ]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [ ]:
vectorstore.embeddings

In [ ]:
vectorstore.similarity_search("similarity",k=5) #Complete the code to pass a query and an appropriate k value

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5} #Complete the code to pass an appropriate k value
)

### Response Function

In [ ]:
qna_system_message = """
You are a clinical decision support assistant. Your task is to assist healthcare professionals by retrieving and synthesizing relevant, up-to-date, and trusted medical knowledge from authorized medical manuals, peer-reviewed research papers, and clinical guidelines. When provided with a clinical query, patient case description, or medical scenario:

Retrieve only relevant information from trusted medical sources within the knowledge base.
Synthesize the retrieved information into a clear, concise, and clinically useful response.
Prioritize evidence-based guidance and current standards of care.
Highlight critical considerations for time-sensitive or emergency situations.
Avoid speculation or unsupported claims.
If sufficient information is not available in the retrieved documents, explicitly state:
  "Insufficient information in the knowledge base to provide a reliable recommendation."

Return the output in the following structured JSON format only:

Answer:
[Answer based on context]

Source:
[Source details with page or section]
"""

In [ ]:
qna_user_message_template = """
###Context
Here are some excerpts from GEN AI Research Paper and their sources that are relevant to the Gen AI question mentioned below:
{context}

###Question
{question}
"""

In [ ]:
def rag_response(user_input,k=3,max_tokens=512,temperature=0,top_p=0.95):
    global qna_system_message,qna_user_message_template
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)
    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
rag_response_1 = rag_response(question_1)
rag_response_1

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
rag_response_2 = rag_response(question_2)
rag_response_2

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
rag_response_3 = rag_response(question_3)
rag_response_3

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
rag_response_4 = rag_response(question_4)
rag_response_4

In [ ]:
#Update the results
results_baseline['rag_response'] = [rag_response_1, rag_response_2, rag_response_3, rag_response_4]

#display results
results_baseline.head()

## Output Evaluation

#### **Evaluation 1: Base Prompt Response Evaluation**

In [ ]:
#Define groundedness for the system
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context.

Instructions:
Briefly summarize, in five sentences or less, whether the answer is derived from the context. Use maximum five sentences.

Finally, return the Score in a dictionary format not json and score should be in the range of 1 to 5.
Example {groundedness_score:4}
"""

In [ ]:
#Define relevance for the system
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
Briefly summarize, in five sentences or less, whether the answer is fulfills the metric. Use maximum five sentences.
Finally, return the Score in a dictionary format not json and score should be in the range of 1 to 5.
Example {relevance_score:4}
"""

In [ ]:
#user message template
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
#Define LLM as a Judge function
def generate_ground_relevance_response(user_input,response,  max_tokens= 512, temperature= 0.7, top_p=1):
    global qna_user_message_template

    context_for_query = [doc.page_content for doc in retriever.get_relevant_documents(user_input, k=5)]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=response)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=response)}
                [/INST]"""

    response_1 = client.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {"role": "user", "content": groundedness_prompt}
                ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
            )

    response_2 = client.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {"role": "user", "content": relevance_prompt}
                ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
            )

    return response_1.choices[0].message.content,response_2.choices[0].message.content

In [ ]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[0], response=results_baseline.Answer[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
#Question 2
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[1], response=results_baseline.Answer[1], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
#Question 3
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[2], response=results_baseline.Answer[2], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
#Question 4
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[0], response=results_baseline.Answer[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

#### **Evaluation 2: Prompt Engineering Response Evaluation**

In [ ]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[0], response=results_baseline.promt_response[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 2
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[1], response=results_baseline.promt_response[1], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 3
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[2], response=results_baseline.promt_response[2], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 4
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[3], response=results_baseline.promt_response[3], max_tokens=516)
print(ground,end="\n\n")
print(rel)

#### **Evaluation 3: RAG Response Evaluation**

In [ ]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[0], response=results_baseline.rag_response[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 2
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[1], response=results_baseline.rag_response[1], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 3
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[2], response=results_baseline.rag_response[2], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 4
ground,rel = generate_ground_relevance_response(user_input=results_baseline.Question[3], response=results_baseline.rag_response[3], max_tokens=516)
print(ground,end="\n\n")
print(rel)